# LLM 변화구 추천 — 유사 투수 쌍 → Statcast 집계 → Gemini / GPT

**파이프라인에서의 위치**

| 단계 | 담당 | 산출물 |
|---|---|---|
| ① 클러스터링 (K-Means / DBSCAN / 계층 / GMM) | 각 팀원 | `pitcher_clustered.json` = {선수: {연도: {average_velocity, cluster}}} |
| ② 유사 투수 검색 | 2번 팀원 | {입력 투수/연도, 가장 유사한 투수/연도} |
| **③ LLM 변화구 추천 (이 노트북)** | 의진 | `data/processed/output/llm_recommendations/*.json, *.md` |

**흐름**: 투수 쌍 입력 → `data/raw`에서 두 투수의 구종별 수치 계산 → 목표 shape 계산(코드) → 시스템 프롬프트 + one-shot + 실제 요청 → LLM(JSON 응답) → 자동 검증 → 저장

**실행 방법**
1. 레포 루트의 `.env` 파일에 API 키를 적는다 (**코드에 키를 쓰지 말 것**, `.env`는 `.gitignore`에 포함).
   - **무료 Gemini**: `GEMINI_API_KEY=AIza...` (Google AI Studio에서 발급, 기본 모델 `gemini-3.8-flash`)
   - OpenAI GPT로 바꿀 때: `OPENAI_API_KEY=sk-...`를 추가하면 코드 수정 없이 OpenAI로 호출한다 (우선 적용, 기본 모델 `gpt-5-mini`).
   - 두 경우 모두 OpenAI 공식 Python SDK를 사용한다 (Gemini는 OpenAI 호환 주소로 접속).
2. 아래 **실행 설정** 셀에서 투수 쌍을 입력한다.
3. 처음에는 `DRY_RUN = True`로 Run All → 프롬프트 확인 (API 호출 없음, 비용 0) → 문제없으면 `False`로 바꿔 다시 실행.

## 0. 환경 설정

In [45]:
from __future__ import annotations

import json
import os
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

try:
    from IPython.display import Markdown, display
except ImportError:
    display, Markdown = print, str

pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 2)

### 실행 설정 (여기만 바꾸면 됨)
- 투수는 **이름**(`"Shohei Ohtani"`, `"Ohtani, Shohei"` 모두 가능) 또는 **MLBAM ID**로 입력
- 2번 단계 결과가 JSON 파일로 있으면 `PAIR_JSON`에 레포 루트 기준 상대경로를 넣으면 아래 직접 입력값보다 우선
- 클러스터 JSON이 여러 개(기법별)면 `CLUSTERS_JSON`으로 사용할 파일을 지정

In [ ]:
# --- 직접 입력 ---
INPUT_PLAYER = "Shohei Ohtani"
INPUT_YEAR = 2023          # 오타니는 2024년에 투구하지 않음
SIMILAR_PLAYER = "Justin Verlander"
SIMILAR_YEAR = 2024

# --- 2번 단계 결과 파일 (있으면 우선 사용, 없으면 None) ---
PAIR_JSON = None          # 예: "data/processed/output/similar_pair.json"

# --- 클러스터 JSON (None이면 data/ 아래 pitcher_clustered.json 자동 탐색) ---
CLUSTERS_JSON = None      # 예: "data/processed/output/pitcher_clustered.json"

DRY_RUN = True            # True: API 호출 없이 프롬프트만 확인 / False: 실제 GPT 호출
FORCE_RERUN = False       # True: 구종 집계 캐시를 무시하고 raw부터 다시 계산
MODEL = None              # None이면 .env의 LLM_MODEL, 없으면 Gemini: gemini-3.8-flash / OpenAI: gpt-5-mini

## 1. 경로와 설정 (레포 루트 기준 상대경로)

In [47]:
def find_repo_root() -> Path:
    """현재 파일 또는 실행 위치에서 위로 올라가며 .git 또는 data 폴더가 있는 곳을 레포 루트로 본다."""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() or (candidate / "data").is_dir():
            return candidate
    return Path.cwd().resolve()


ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "processed" / "interim"
ARSENAL_PKL = INTERIM_DIR / "pitch_arsenal.pkl"


def rel(p: Path) -> str:
    """출력용 상대경로 (공개 레포라 절대경로/사용자명 노출 방지)."""
    try:
        return Path(p).resolve().relative_to(ROOT).as_posix()
    except ValueError:
        return Path(p).name

In [48]:
TARGET_YEARS = [2021, 2022, 2023, 2024, 2025]
FASTBALL_TYPES = ["FF", "SI", "FC"]          # 팀 공통: 주 패스트볼 후보
EXCLUDE_PITCH_TYPES = {"PO", "FA", "EP", "CS", "KN", "SC", "IN", "UN", "AB"}  # 피치아웃/이퓨스 등
MIN_PITCHES_TO_LIST = 20                      # 이보다 적게 던진 구종은 프롬프트에서 제외
SMALL_SAMPLE_PITCHES = 100                    # 이보다 적으면 '표본 작음' 표시
SMALL_SAMPLE_BBE = 30                         # 인플레이 타구가 이보다 적으면 GB%/xwOBAcon 불안정

USECOLS = {
    "pitcher", "player_name", "game_year", "game_type", "pitch_type", "pitch_name",
    "description", "zone", "stand", "p_throws",
    "release_speed", "pfx_x", "pfx_z", "release_spin_rate",
    "release_extension", "arm_angle", "release_pos_x", "release_pos_z",
    "delta_run_exp",
    "bb_type", "estimated_woba_using_speedangle",   # GB%, xwOBAcon용
}

SWING_DESC = {
    "swinging_strike", "swinging_strike_blocked", "foul", "foul_tip",
    "hit_into_play", "foul_bunt", "missed_bunt", "bunt_foul_tip",
}
WHIFF_DESC = {"swinging_strike", "swinging_strike_blocked", "missed_bunt"}
CALLED_STRIKE_DESC = {"called_strike"}

## 2. raw Statcast 로드 → 투수-시즌-구종별 집계

- HB는 **암사이드 +**로 좌/우완 통일, RV/100은 **투수 기준 + = 좋음**
- GB%, xwOBAcon은 인플레이 타구(BBE) 기준 (계획서 6절 구종 역할별 지표)

In [49]:
def load_raw_pitches(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    files = [f for f in sorted(raw_dir.rglob("*.csv"))
             if not f.name.startswith(("~$", "."))]
    if not files:
        raise FileNotFoundError(
            f"{rel(raw_dir)} 안에 csv가 없습니다. Baseball Savant pitch-level CSV를 넣어주세요."
        )
    frames = []
    for f in files:
        try:
            frames.append(pd.read_csv(f, low_memory=False, usecols=lambda c: c in USECOLS))
        except Exception as e:  # 깨진 파일은 건너뜀
            print(f"[arsenal] 읽기 실패, 건너뜀: {rel(f)} ({e})")
    data = pd.concat(frames, ignore_index=True)

    num_cols = ["game_year", "zone", "release_speed", "pfx_x", "pfx_z", "release_spin_rate",
                "release_extension", "arm_angle", "release_pos_x", "release_pos_z", "delta_run_exp",
                "estimated_woba_using_speedangle"]
    for c in num_cols:
        if c in data.columns:
            data[c] = pd.to_numeric(data[c], errors="coerce")

    mask = (
        (data["game_type"] == "R")
        & data["game_year"].isin(TARGET_YEARS)
        & data["p_throws"].isin(["R", "L"])
        & data["pitch_type"].notna()
        & ~data["pitch_type"].isin(EXCLUDE_PITCH_TYPES)
        & (data["description"] != "pitchout")
    )
    data = data.loc[mask].copy()
    data["game_year"] = data["game_year"].astype(int)
    print(f"[arsenal] raw 파일 {len(frames)}개, 정규시즌 투구 {len(data):,}개")
    return data

In [50]:
def _add_pitch_flags(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    hand = d["p_throws"].map({"R": -1.0, "L": 1.0})
    d["ivb_in"] = d["pfx_z"] * 12.0
    d["hb_in"] = d["pfx_x"] * 12.0 * hand          # 암사이드 양수
    d["is_swing"] = d["description"].isin(SWING_DESC)
    d["is_whiff"] = d["description"].isin(WHIFF_DESC)
    d["is_csw"] = d["description"].isin(WHIFF_DESC | CALLED_STRIKE_DESC)
    if "zone" in d.columns:
        d["is_out_zone"] = d["zone"].between(11, 14)
    else:
        d["is_out_zone"] = False
    d["is_chase"] = d["is_out_zone"] & d["is_swing"]
    # delta_run_exp는 타격팀 기준 → 투수 기준으로 부호 반전
    d["rv_pitcher"] = -d["delta_run_exp"] if "delta_run_exp" in d.columns else np.nan
    # 인플레이 타구(BBE) 지표: GB% = 땅볼/BBE, xwOBAcon = BBE의 기대 wOBA 평균
    bb = d["bb_type"] if "bb_type" in d.columns else pd.Series(np.nan, index=d.index)
    d["is_bbe"] = bb.notna()
    d["is_gb"] = bb == "ground_ball"
    d["xwoba_bbe"] = (d["estimated_woba_using_speedangle"].where(d["is_bbe"])
                      if "estimated_woba_using_speedangle" in d.columns else np.nan)
    return d


def build_arsenal_table(data: pd.DataFrame) -> pd.DataFrame:
    """투수-시즌-구종 단위 요약표."""
    d = _add_pitch_flags(data)
    keys = ["pitcher", "game_year", "pitch_type"]

    g = d.groupby(keys, observed=True)
    ars = g.agg(
        player_name=("player_name", "first"),
        p_throws=("p_throws", "first"),
        pitch_name=("pitch_name", "first") if "pitch_name" in d.columns else ("pitch_type", "first"),
        n=("pitch_type", "size"),
        velo_mph=("release_speed", "mean"),
        ivb_in=("ivb_in", "mean"),
        hb_in=("hb_in", "mean"),
        spin_rpm=("release_spin_rate", "mean"),
        extension_ft=("release_extension", "mean"),
        arm_angle_deg=("arm_angle", "mean"),
        release_x_ft=("release_pos_x", "mean"),
        release_z_ft=("release_pos_z", "mean"),
        swings=("is_swing", "sum"),
        whiffs=("is_whiff", "sum"),
        csw=("is_csw", "sum"),
        out_zone=("is_out_zone", "sum"),
        chases=("is_chase", "sum"),
        rv_sum=("rv_pitcher", "sum"),
        rv_n=("rv_pitcher", "count"),
        n_bbe=("is_bbe", "sum"),
        gbs=("is_gb", "sum"),
        xwobacon=("xwoba_bbe", "mean"),
    ).reset_index()

    # 비율 지표
    ars["whiff_pct"] = 100 * ars["whiffs"] / ars["swings"].replace(0, np.nan)
    ars["csw_pct"] = 100 * ars["csw"] / ars["n"]
    ars["chase_pct"] = 100 * ars["chases"] / ars["out_zone"].replace(0, np.nan)
    ars["rv_per_100"] = 100 * ars["rv_sum"] / ars["rv_n"].replace(0, np.nan)
    ars["gb_pct"] = 100 * ars["gbs"] / ars["n_bbe"].replace(0, np.nan)

    # 구사율 (전체 / 좌타 / 우타)
    total = d.groupby(["pitcher", "game_year"]).size().rename("season_total")
    ars = ars.merge(total, on=["pitcher", "game_year"])
    ars["usage_pct"] = 100 * ars["n"] / ars["season_total"]
    for side in ("L", "R"):
        ds = d[d["stand"] == side]
        side_n = ds.groupby(keys, observed=True).size().rename("_side_n").reset_index()
        side_total = ds.groupby(["pitcher", "game_year"]).size().rename("_side_total").reset_index()
        side_n = side_n.merge(side_total, on=["pitcher", "game_year"])
        side_n[f"usage_vs_{side}HH_pct"] = 100 * side_n["_side_n"] / side_n["_side_total"]
        ars = ars.merge(side_n[[*keys, f"usage_vs_{side}HH_pct"]], on=keys, how="left")

    # 주 패스트볼(FF/SI/FC 중 최다) 및 패스트볼 대비 차이
    fb = ars[ars["pitch_type"].isin(FASTBALL_TYPES)]
    primary = fb.loc[fb.groupby(["pitcher", "game_year"])["n"].idxmax(),
                     ["pitcher", "game_year", "pitch_type", "velo_mph", "ivb_in", "hb_in"]]
    primary = primary.rename(columns={"pitch_type": "primary_fastball",
                                      "velo_mph": "_fb_velo", "ivb_in": "_fb_ivb", "hb_in": "_fb_hb"})
    ars = ars.merge(primary, on=["pitcher", "game_year"], how="left")
    ars["is_primary_fastball"] = ars["pitch_type"] == ars["primary_fastball"]
    ars["velo_gap_vs_fb"] = ars["velo_mph"] - ars["_fb_velo"]
    ars["ivb_gap_vs_fb"] = ars["ivb_in"] - ars["_fb_ivb"]
    ars["hb_gap_vs_fb"] = ars["hb_in"] - ars["_fb_hb"]
    ars["small_sample"] = ars["n"] < SMALL_SAMPLE_PITCHES

    drop = ["gbs", "swings", "whiffs", "csw", "out_zone", "chases", "rv_sum", "rv_n", "_fb_velo", "_fb_ivb", "_fb_hb"]
    return ars.drop(columns=drop).sort_values(["pitcher", "game_year", "n"],
                                              ascending=[True, True, False]).reset_index(drop=True)


def load_arsenal(force_rerun: bool = False) -> pd.DataFrame:
    """캐시가 있으면 불러오고, 없으면 raw에서 계산 후 저장."""
    if ARSENAL_PKL.exists() and not force_rerun:
        cached = pd.read_pickle(ARSENAL_PKL)
        if {"gb_pct", "xwobacon", "n_bbe"}.issubset(cached.columns):   # 예전 버전 캐시면 다시 계산
            return cached
    ars = build_arsenal_table(load_raw_pitches())
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    ars.to_pickle(ARSENAL_PKL)
    print(f"[arsenal] 저장: {rel(ARSENAL_PKL)} ({len(ars):,}행)")
    return ars

## 3. 선수 찾기
이름 표기 차이(악센트, `Last, First`, Jr.)를 맞추고, 동명이인은 평균구속으로 구분한다.

In [51]:
def _norm_name(s: str) -> str:
    """대소문자·악센트(é→e)·마침표·접미사(Jr./Sr./II/III) 차이를 무시하고 비교하기 위한 정규화."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.replace(".", "").replace("-", " ").lower()
    tokens = [t for t in s.replace(",", " , ").split() if t not in {"jr", "sr", "ii", "iii", "iv"}]
    return " ".join(tokens).replace(" ,", ",")


def _name_variants(n: str) -> set:
    """Statcast 'Last, First' 표기와 pitcher_clustered.json의 'First Last' 표기를 모두 만든다."""
    n = _norm_name(n)
    out = {n}
    if "," in n:
        last, first = [x.strip() for x in n.split(",", 1)]
        out |= {f"{first} {last}", f"{last} {first}"}
    return out


def resolve_pitcher(arsenal: pd.DataFrame, player, year: int, velo_hint: float | None = None) -> int:
    """MLBAM ID(숫자) 또는 이름('Shohei Ohtani' / 'Ohtani, Shohei')으로 투수 ID를 찾는다.

    동명이인(예: Luis García)이 있으면 velo_hint(pitcher_clustered.json의 average_velocity)와
    주 패스트볼 평균구속이 가장 가까운 투수를 고른다.
    """
    season = arsenal[arsenal["game_year"] == int(year)]
    if season.empty:
        raise ValueError(f"{year} 시즌 데이터가 없습니다.")

    if str(player).strip().isdigit():
        pid = int(player)
        if pid not in set(season["pitcher"]):
            raise ValueError(f"투수 ID {pid}의 {year} 시즌 데이터가 없습니다.")
        return pid

    target = _norm_name(player)
    names = season.drop_duplicates("pitcher")[["pitcher", "player_name"]]
    hits = names[names["player_name"].map(lambda n: target in _name_variants(n))]

    if len(hits) == 1:
        return int(hits.iloc[0]["pitcher"])
    if len(hits) > 1:
        fb = season[season["is_primary_fastball"] & season["pitcher"].isin(hits["pitcher"])]
        if velo_hint is not None and not fb.empty:
            best = fb.loc[(fb["velo_mph"] - float(velo_hint)).abs().idxmin()]
            print(f"[resolve] '{player}' {year} 동명이인 {len(hits)}명 → 평균구속 {velo_hint}mph에 "
                  f"가장 가까운 ID {int(best['pitcher'])} ({best['velo_mph']:.1f}mph) 선택")
            return int(best["pitcher"])
        raise ValueError(f"'{player}' {year} 동명이인 {len(hits)}명: {hits.to_dict('records')} → ID로 입력하세요.")
    # 다른 시즌에는 있는지 확인해서 안내 (예: 2024 오타니처럼 그 해에 투구하지 않은 경우)
    all_names = arsenal.drop_duplicates(["pitcher", "game_year"])[["pitcher", "player_name", "game_year"]]
    other = all_names[all_names["player_name"].map(lambda n: target in _name_variants(n))]
    if not other.empty:
        years = sorted(other["game_year"].unique().tolist())
        raise ValueError(f"'{player}'는 {year} 시즌 투구 데이터가 없습니다. 데이터가 있는 시즌: {years}")
    raise ValueError(f"'{player}'({year})를 찾을 수 없습니다. 이름 표기(예: 'Gerrit Cole') 또는 MLBAM ID를 확인하세요.")

## 4. 프롬프트에 넣을 요약과 목표 shape
목표 shape = 입력 투수 주 패스트볼 + 유사 투수의 (변화구 − 패스트볼) 차이. GPT가 산술을 틀리지 않도록 코드가 계산한다.

In [52]:
def _r(x, nd=1):
    return None if pd.isna(x) else round(float(x), nd)


def pitcher_season_summary(arsenal: pd.DataFrame, pitcher_id: int, year: int) -> dict:
    """한 투수-시즌을 프롬프트에 넣을 dict로 요약한다."""
    rows = arsenal[(arsenal["pitcher"] == pitcher_id) & (arsenal["game_year"] == int(year))]
    if rows.empty:
        raise ValueError(f"투수 {pitcher_id}의 {year} 시즌 데이터가 없습니다.")
    rows = rows[rows["n"] >= MIN_PITCHES_TO_LIST]

    fb = rows[rows["is_primary_fastball"]]
    fb = fb.iloc[0] if not fb.empty else None
    first = rows.iloc[0]

    pitches = []
    for _, r in rows.iterrows():
        pitches.append({
            "pitch_type": r["pitch_type"],
            "pitch_name": r["pitch_name"],
            "is_primary_fastball": bool(r["is_primary_fastball"]),
            "n_pitches": int(r["n"]),
            "small_sample": bool(r["small_sample"]),
            "usage_pct": _r(r["usage_pct"]),
            "usage_vs_LHH_pct": _r(r.get("usage_vs_LHH_pct")),
            "usage_vs_RHH_pct": _r(r.get("usage_vs_RHH_pct")),
            "velo_mph": _r(r["velo_mph"]),
            "ivb_in": _r(r["ivb_in"]),
            "hb_in": _r(r["hb_in"]),
            "spin_rpm": _r(r["spin_rpm"], 0),
            "velo_gap_vs_fb": _r(r["velo_gap_vs_fb"]),
            "ivb_gap_vs_fb": _r(r["ivb_gap_vs_fb"]),
            "hb_gap_vs_fb": _r(r["hb_gap_vs_fb"]),
            "whiff_pct": _r(r["whiff_pct"]),
            "csw_pct": _r(r["csw_pct"]),
            "chase_pct": _r(r["chase_pct"]),
            "rv_per_100": _r(r["rv_per_100"], 2),
            "n_bbe": int(r["n_bbe"]),
            "gb_pct": _r(r["gb_pct"]),
            "xwobacon": _r(r["xwobacon"], 3),
        })

    return {
        "pitcher_id": int(pitcher_id),
        "player_name": first["player_name"],
        "season": int(year),
        "throws": first["p_throws"],
        "primary_fastball": None if fb is None else {
            "pitch_type": fb["pitch_type"],
            "velo_mph": _r(fb["velo_mph"]),
            "ivb_in": _r(fb["ivb_in"]),
            "hb_in": _r(fb["hb_in"]),
            "spin_rpm": _r(fb["spin_rpm"], 0),
            "arm_angle_deg": _r(fb["arm_angle_deg"]),
            "extension_ft": _r(fb["extension_ft"]),
            "release_height_ft": _r(fb["release_z_ft"]),
            "release_side_ft": _r(abs(fb["release_x_ft"]) if pd.notna(fb["release_x_ft"]) else np.nan),
        },
        "arsenal": pitches,
    }


def transfer_targets(input_summary: dict, similar_summary: dict) -> list[dict]:
    """유사 투수의 (변화구 − 패스트볼) 차이를 입력 투수의 패스트볼에 더해 목표 shape를 계산한다.

    GPT가 산술을 틀리지 않도록 코드에서 미리 계산해 넘긴다.
    """
    fb_in = input_summary.get("primary_fastball")
    if fb_in is None or None in (fb_in["velo_mph"], fb_in["ivb_in"], fb_in["hb_in"]):
        return []
    input_types = {p["pitch_type"]: p for p in input_summary["arsenal"]}

    out = []
    for p in similar_summary["arsenal"]:
        if p["is_primary_fastball"] or None in (p["velo_gap_vs_fb"], p["ivb_gap_vs_fb"], p["hb_gap_vs_fb"]):
            continue
        target = {
            "pitch_type": p["pitch_type"],
            "pitch_name": p["pitch_name"],
            "source_usage_pct": p["usage_pct"],
            "source_n_pitches": p["n_pitches"],
            "source_small_sample": p["small_sample"],
            "gap_vs_fb": {"velo_mph": p["velo_gap_vs_fb"], "ivb_in": p["ivb_gap_vs_fb"], "hb_in": p["hb_gap_vs_fb"]},
            "target_shape": {
                "velo_mph": round(fb_in["velo_mph"] + p["velo_gap_vs_fb"], 1),
                "ivb_in": round(fb_in["ivb_in"] + p["ivb_gap_vs_fb"], 1),
                "hb_in": round(fb_in["hb_in"] + p["hb_gap_vs_fb"], 1),
            },
            "input_already_throws": p["pitch_type"] in input_types,
        }
        if target["input_already_throws"]:
            cur = input_types[p["pitch_type"]]
            target["input_current_shape"] = {"velo_mph": cur["velo_mph"], "ivb_in": cur["ivb_in"], "hb_in": cur["hb_in"]}
        out.append(target)
    return out

## 5. 프롬프트 (시스템 프롬프트 + one-shot)

**설계 원칙**

1. 근거 고정(grounding): GPT는 우리가 넘긴 Statcast 집계 수치만 사용한다. 선수에 대한 사전 지식이나
   외부 정보(부상, 뉴스, 다른 시즌 기록)는 쓰지 않는다 → 수치 지어내기 방지.
2. 후보 제한: 추천 구종은 '유사 투수가 실제로 던진 구종' 또는 '입력 투수가 이미 던지는 구종의 조정'만 허용.
3. 산술은 코드가: 목표 shape(입력 FB + 유사 투수의 변화구−FB 차이)는 코드가 미리 계산해 transfer_targets로 준다.
   GPT는 이를 인용·선택·해석만 한다.
4. 고정 형식: JSON 스키마를 강제해 결과를 저장·비교·렌더링할 수 있게 한다.
5. 관측 연구의 한계 명시: 인과 주장("이 구종을 배우면 FIP가 내려간다") 금지, 표본 크기 경고.
6. one-shot 예시는 가상의 투수(예시 투수 A/B)로 만든다 → 실제 선수 정보가 답변에 섞이는 것 방지.
7. 연구계획서 6절: 주 성과지표는 RV/100, 구종 역할별로 메커니즘 지표를 다르게 본다
   (스위퍼·슬라이더: Whiff·Chase / 체인지업·스플리터: Whiff·GB·xwOBAcon / 커터: Whiff·약한 컨택).
8. 연구계획서 10절: 성공 사례뿐 아니라 실패 사례(negative control)도 함께 제시한다.

In [53]:
OUTPUT_SCHEMA = {
    "summary": "string — 2~3문장. 두 투수 패스트볼의 공통점과 핵심 추천을 요약",
    "fastball_comparison": {
        "similarities": ["string — 수치 포함 (예: '팔 각도 45.9° vs 44.1°')"],
        "differences": ["string — 수치 포함"],
    },
    "recommendations": [
        {
            "rank": "integer 1~3",
            "pitch_type": "string — Statcast 코드 (예: ST, SL, CH). 반드시 입력 데이터에 존재하는 코드",
            "pitch_name": "string",
            "action": "'add'(새로 추가) | 'refine'(이미 던지는 구종의 shape 조정) | 'usage'(구사율/사용 상황 조정)",
            "target_shape": {"velo_mph": "number", "ivb_in": "number", "hb_in": "number"},
            "gap_vs_fastball": {"velo_mph": "number", "ivb_in": "number", "hb_in": "number"},
            "rationale": ["string — 근거 2~4개, 각 근거에 입력 데이터의 수치를 인용"],
            "usage_plan": "string — 어떤 타자(좌/우)·상황에 쓰는지, 유사 투수의 좌/우타 구사율 근거",
            "confidence": "'high' | 'medium' | 'low'",
            "caveats": ["string — 표본 크기, 구현 난이도 등"],
        }
    ],
    "failure_cases": [
        {"pitcher": "'input' | 'similar'", "pitch_type": "string",
         "lesson": "string — 성과가 나빴던 구종(RV/100 음수, xwOBAcon 높음 등)에서 피해야 할 shape/사용법 (수치 인용)"}
    ],
    "not_recommended": [
        {"pitch_type": "string", "reason": "string — 유사 투수가 던졌지만 추천하지 않는 이유 (수치 인용)"}
    ],
    "data_limitations": ["string — 이 추천의 한계"],
}

SYSTEM_PROMPT = f"""당신은 MLB 투구 설계(pitch design) 분석가입니다.
대학 연구 프로젝트 '주 패스트볼 shape 기반 개인화 변화구 추천'의 마지막 단계에서,
[입력 투수]와 패스트볼이 비슷하지만 성적이 더 좋았던 [유사 투수]의 Statcast 집계 데이터를 비교해
입력 투수에게 맞는 변화구(세컨더리 피치) shape 목표를 최대 3개 제시합니다.

## 입력 데이터 정의
- 모든 수치는 해당 시즌 MLB 정규시즌 Statcast 투구 데이터에서 계산한 값입니다.
- velo_mph: 평균 구속(mph). ivb_in: 수직 무브먼트(인치, 중력 제외, +는 떠오름).
- hb_in: 수평 무브먼트(인치). **암사이드(투수 팔 쪽) = +, 글러브사이드 = −** 로 좌/우완 통일.
- *_gap_vs_fb: 같은 투수의 주 패스트볼 대비 차이(해당 구종 − 주 패스트볼).
- whiff_pct: 헛스윙/스윙(%). chase_pct: 존 밖 공에 대한 스윙 비율(%).
- primary_fastball: 주 패스트볼의 릴리스 기하(팔 각도, 익스텐션, 릴리스 높이/좌우). 구속·무브먼트는 arsenal의 해당 구종 참고.
- rv_per_100: 100구당 투수 기준 런 밸류. **+일수록 투수에게 좋음**. (주 성과지표)
- gb_pct: 인플레이 타구(BBE) 중 땅볼 비율(%). xwobacon: BBE의 기대 wOBA 평균, **낮을수록 투수에게 좋음** (리그 평균 약 0.370 내외).
- n_bbe: 인플레이 타구 수. 30개 미만이면 gb_pct·xwobacon은 불안정합니다.
- small_sample=true: 100구 미만 → 비율 지표가 불안정함.
- transfer_targets: 코드가 미리 계산한 목표 shape = 입력 투수 주 패스트볼 + 유사 투수의 (구종 − 패스트볼) 차이.
- search_context: 유사 투수 검색 단계에서 넘어온 정보(클러스터, 거리, FIP 등). 없을 수도 있습니다.

## 반드시 지킬 규칙
1. 입력 JSON에 있는 수치만 사용하세요. 선수에 대한 사전 지식, 다른 시즌 기록, 부상·뉴스·코칭 정보는 쓰지 마세요.
   선수 이름은 식별용일 뿐입니다.
2. 추천 구종(pitch_type)은 (a) 유사 투수의 arsenal에 있는 구종, 또는 (b) 입력 투수가 이미 던지는 구종의 조정만 허용됩니다.
   데이터에 없는 구종을 만들지 마세요.
3. target_shape와 gap_vs_fastball은 transfer_targets의 값을 그대로 사용하세요. 직접 새로 계산하지 마세요.
   action이 'usage'면 입력 투수의 현재 shape를 target_shape에 넣으세요.
4. 각 rationale 항목에는 입력 데이터의 숫자를 최소 1개 인용하세요 (예: "유사 투수 ST whiff 38.2%, RV/100 +1.9").
5. 추천 우선순위:
   (1) 성과: 유사 투수에게서 rv_per_100이 좋았던 구종
   (2) 역할별 메커니즘: 구종 역할에 맞는 지표로 성공 이유를 확인하세요.
       - 스위퍼·슬라이더·커브(ST, SL, SV, CU, KC): whiff_pct, chase_pct
       - 체인지업·스플리터(CH, FS, FO): whiff_pct, gb_pct, xwobacon
       - 커터·싱커(FC, SI): whiff_pct, xwobacon(약한 컨택)
   (3) 분리: 입력 투수 패스트볼과의 구속·무브먼트 차이
   (4) 역할 보완: 입력 투수에게 없거나 성과가 나쁜 역할(예: 반대손 타자 대응 구종)을 채우는지
5-1. 실패 사례: 유사 투수 또는 입력 투수의 구종 중 rv_per_100이 음수이거나 xwobacon이 높은 구종이 있으면
   failure_cases에 "어떤 shape/사용법을 피해야 하는지"를 수치와 함께 1~2개 쓰세요. 해당 사례가 없으면 빈 리스트로 두세요.
6. 표본이 작거나(small_sample, 또는 n_bbe 30 미만인데 gb_pct·xwobacon을 근거로 쓸 때) 구사율이 5% 미만인 구종은 confidence를 'low'로 두고 caveats에 이유를 쓰세요.
7. 관측 데이터 기반 추천입니다. "이 구종을 익히면 성적이 오른다" 같은 인과적 단정은 하지 말고
   "유사한 패스트볼을 가진 투수에게서 효과적이었다" 수준으로 표현하세요. 부상 위험·그립 등 생체역학은 판단하지 마세요.
8. 근거가 부족하면 추천을 3개보다 적게 내도 됩니다. 억지로 채우지 마세요.
9. 한국어로 작성하되 구종 코드(FF, SL, ST 등)와 지표명(IVB, HB, Whiff%)은 영어 그대로 쓰세요.

## 출력 형식
아래 스키마를 따르는 JSON 객체 하나만 출력하세요. 코드 블록(```)이나 다른 설명 문장은 붙이지 마세요.
{json.dumps(OUTPUT_SCHEMA, ensure_ascii=False, indent=2)}
"""

### one-shot 예시 (가상의 투수 A/B — 실제 선수 아님)
좋은 답의 기준을 보여준다: 모든 근거에 수치 인용, 역할별 지표 사용, 실패 사례 제시, 표본 작은 구종은 제외.

In [54]:
ONE_SHOT_USER = {
    "task": "입력 투수에게 맞는 변화구 shape 목표를 최대 3개 추천",
    "search_context": {"cluster": 0, "fastball_distance": 0.42, "input_fip": 4.61, "similar_fip": 3.38},
    "input_pitcher": {
        "player_name": "예시 투수 A (가상)", "season": 2024, "throws": "R",
        "primary_fastball": {"pitch_type": "FF", "velo_mph": 94.0, "ivb_in": 15.8, "hb_in": 7.5,
                             "spin_rpm": 2350, "arm_angle_deg": 44.0, "extension_ft": 6.4,
                             "release_height_ft": 5.9, "release_side_ft": 1.9},
        "arsenal": [
            {"pitch_type": "FF", "pitch_name": "4-Seam Fastball", "is_primary_fastball": True, "n_pitches": 1180,
             "small_sample": False, "usage_pct": 55.0, "usage_vs_LHH_pct": 57.0, "usage_vs_RHH_pct": 53.0,
             "velo_mph": 94.0, "ivb_in": 15.8, "hb_in": 7.5, "spin_rpm": 2350,
             "velo_gap_vs_fb": 0.0, "ivb_gap_vs_fb": 0.0, "hb_gap_vs_fb": 0.0,
             "whiff_pct": 21.0, "csw_pct": 27.5, "chase_pct": 22.0, "rv_per_100": -0.4},
            {"pitch_type": "SL", "pitch_name": "Slider", "is_primary_fastball": False, "n_pitches": 640,
             "small_sample": False, "usage_pct": 30.0, "usage_vs_LHH_pct": 18.0, "usage_vs_RHH_pct": 41.0,
             "velo_mph": 86.5, "ivb_in": 3.0, "hb_in": -3.5, "spin_rpm": 2400,
             "velo_gap_vs_fb": -7.5, "ivb_gap_vs_fb": -12.8, "hb_gap_vs_fb": -11.0,
             "whiff_pct": 29.0, "csw_pct": 29.0, "chase_pct": 30.0, "rv_per_100": 0.3},
            {"pitch_type": "CH", "pitch_name": "Changeup", "is_primary_fastball": False, "n_pitches": 320,
             "small_sample": False, "usage_pct": 15.0, "usage_vs_LHH_pct": 25.0, "usage_vs_RHH_pct": 6.0,
             "velo_mph": 87.5, "ivb_in": 9.5, "hb_in": 12.5, "spin_rpm": 1800,
             "velo_gap_vs_fb": -6.5, "ivb_gap_vs_fb": -6.3, "hb_gap_vs_fb": 5.0,
             "whiff_pct": 24.0, "csw_pct": 24.0, "chase_pct": 26.0, "rv_per_100": -1.6},
        ],
    },
    "similar_pitcher": {
        "player_name": "예시 투수 B (가상)", "season": 2023, "throws": "R",
        "primary_fastball": {"pitch_type": "FF", "velo_mph": 94.6, "ivb_in": 16.4, "hb_in": 7.0,
                             "spin_rpm": 2380, "arm_angle_deg": 45.5, "extension_ft": 6.5,
                             "release_height_ft": 6.0, "release_side_ft": 1.8},
        "arsenal": [
            {"pitch_type": "FF", "pitch_name": "4-Seam Fastball", "is_primary_fastball": True, "n_pitches": 1100,
             "small_sample": False, "usage_pct": 48.0, "usage_vs_LHH_pct": 50.0, "usage_vs_RHH_pct": 46.0,
             "velo_mph": 94.6, "ivb_in": 16.4, "hb_in": 7.0, "spin_rpm": 2380,
             "velo_gap_vs_fb": 0.0, "ivb_gap_vs_fb": 0.0, "hb_gap_vs_fb": 0.0,
             "whiff_pct": 24.0, "csw_pct": 29.0, "chase_pct": 24.0, "rv_per_100": 0.8},
            {"pitch_type": "ST", "pitch_name": "Sweeper", "is_primary_fastball": False, "n_pitches": 560,
             "small_sample": False, "usage_pct": 25.0, "usage_vs_LHH_pct": 12.0, "usage_vs_RHH_pct": 38.0,
             "velo_mph": 83.0, "ivb_in": 1.0, "hb_in": -14.0, "spin_rpm": 2650,
             "velo_gap_vs_fb": -11.6, "ivb_gap_vs_fb": -15.4, "hb_gap_vs_fb": -21.0,
             "whiff_pct": 37.0, "csw_pct": 33.0, "chase_pct": 33.0, "rv_per_100": 1.9},
            {"pitch_type": "FS", "pitch_name": "Split-Finger", "is_primary_fastball": False, "n_pitches": 390,
             "small_sample": False, "usage_pct": 17.0, "usage_vs_LHH_pct": 29.0, "usage_vs_RHH_pct": 6.0,
             "velo_mph": 86.0, "ivb_in": 4.5, "hb_in": 10.0, "spin_rpm": 1350,
             "velo_gap_vs_fb": -8.6, "ivb_gap_vs_fb": -11.9, "hb_gap_vs_fb": 3.0,
             "whiff_pct": 36.0, "csw_pct": 30.0, "chase_pct": 35.0, "rv_per_100": 1.4},
            {"pitch_type": "CU", "pitch_name": "Curveball", "is_primary_fastball": False, "n_pitches": 70,
             "small_sample": True, "usage_pct": 3.0, "usage_vs_LHH_pct": 4.0, "usage_vs_RHH_pct": 2.0,
             "velo_mph": 79.0, "ivb_in": -9.0, "hb_in": -8.0, "spin_rpm": 2700,
             "velo_gap_vs_fb": -15.6, "ivb_gap_vs_fb": -25.4, "hb_gap_vs_fb": -15.0,
             "whiff_pct": 30.0, "csw_pct": 31.0, "chase_pct": 20.0, "rv_per_100": 2.5},
        ],
    },
    "transfer_targets": [
        {"pitch_type": "ST", "pitch_name": "Sweeper", "source_usage_pct": 25.0, "source_n_pitches": 560,
         "source_small_sample": False, "gap_vs_fb": {"velo_mph": -11.6, "ivb_in": -15.4, "hb_in": -21.0},
         "target_shape": {"velo_mph": 82.4, "ivb_in": 0.4, "hb_in": -13.5}, "input_already_throws": False},
        {"pitch_type": "FS", "pitch_name": "Split-Finger", "source_usage_pct": 17.0, "source_n_pitches": 390,
         "source_small_sample": False, "gap_vs_fb": {"velo_mph": -8.6, "ivb_in": -11.9, "hb_in": 3.0},
         "target_shape": {"velo_mph": 85.4, "ivb_in": 3.9, "hb_in": 10.5}, "input_already_throws": False},
        {"pitch_type": "CU", "pitch_name": "Curveball", "source_usage_pct": 3.0, "source_n_pitches": 70,
         "source_small_sample": True, "gap_vs_fb": {"velo_mph": -15.6, "ivb_in": -25.4, "hb_in": -15.0},
         "target_shape": {"velo_mph": 78.4, "ivb_in": -9.6, "hb_in": -7.5}, "input_already_throws": False},
    ],
}

# one-shot 예시 투구에 BBE 지표 추가 (가상 수치): (n_bbe, gb_pct, xwobacon)
_EXAMPLE_BBE = {
    ("input_pitcher", "FF"): (210, 33.0, 0.395), ("input_pitcher", "SL"): (105, 40.0, 0.350),
    ("input_pitcher", "CH"): (70, 38.0, 0.430),
    ("similar_pitcher", "FF"): (190, 31.0, 0.360), ("similar_pitcher", "ST"): (80, 42.0, 0.310),
    ("similar_pitcher", "FS"): (65, 55.0, 0.300), ("similar_pitcher", "CU"): (12, 50.0, 0.280),
}
for _who in ("input_pitcher", "similar_pitcher"):
    for _p in ONE_SHOT_USER[_who]["arsenal"]:
        _p["n_bbe"], _p["gb_pct"], _p["xwobacon"] = _EXAMPLE_BBE[(_who, _p["pitch_type"])]

ONE_SHOT_ASSISTANT = {
    "summary": "두 투수는 94mph대 라이징 포심(IVB 15.8 vs 16.4in)과 비슷한 팔 각도(44.0° vs 45.5°)를 공유합니다. "
               "유사 투수 B의 성과를 이끈 스위퍼와 스플리터를 입력 투수 A의 패스트볼 기준 목표 shape로 옮기는 것을 우선 추천합니다.",
    "fastball_comparison": {
        "similarities": ["구속 94.0 vs 94.6mph, IVB 15.8 vs 16.4in로 거의 같은 라이징 포심",
                         "팔 각도 44.0° vs 45.5°, 익스텐션 6.4 vs 6.5ft로 릴리스 기하가 유사"],
        "differences": ["포심 RV/100이 A -0.4, B +0.8로 B가 더 효과적",
                        "B는 포심 구사율 48%로 A(55%)보다 변화구 비중이 높음"],
    },
    "recommendations": [
        {"rank": 1, "pitch_type": "ST", "pitch_name": "Sweeper", "action": "add",
         "target_shape": {"velo_mph": 82.4, "ivb_in": 0.4, "hb_in": -13.5},
         "gap_vs_fastball": {"velo_mph": -11.6, "ivb_in": -15.4, "hb_in": -21.0},
         "rationale": ["B의 ST는 whiff 37.0%, RV/100 +1.9로 B 아스날 중 표본이 충분한 구종 가운데 가장 효과적",
                       "A의 현재 SL(HB -3.5in)보다 글러브사이드로 10in 더 휘어 포심과 좌우 분리가 커짐",
                       "B는 우타자 상대 ST 구사율 38%로, A의 우타자 상대 SL(41%) 역할을 대체·보강할 수 있음"],
         "usage_plan": "우타자 상대 주 결정구. B처럼 우타자 상대 30~40% 수준에서 시작",
         "confidence": "high",
         "caveats": ["A의 SL과 그립·회전 방식이 달라 구현 난이도는 데이터로 판단할 수 없음"]},
        {"rank": 2, "pitch_type": "FS", "pitch_name": "Split-Finger", "action": "add",
         "target_shape": {"velo_mph": 85.4, "ivb_in": 3.9, "hb_in": 10.5},
         "gap_vs_fastball": {"velo_mph": -8.6, "ivb_in": -11.9, "hb_in": 3.0},
         "rationale": ["A의 CH는 RV/100 -1.6, xwOBAcon 0.430으로 좌타자 대응 구종이 약함",
                       "B의 FS는 whiff 36.0%, GB 55.0%, xwOBAcon 0.300, RV/100 +1.4이고 좌타자 상대 구사율 29%",
                       "FS 목표 IVB 3.9in는 A의 CH(9.5in)보다 5.6in 더 떨어져 포심과 수직 분리가 큼"],
         "usage_plan": "좌타자 상대 체인지업 대체 구종으로 사용",
         "confidence": "medium",
         "caveats": ["A의 CH를 대체할지 병행할지는 데이터로 결정할 수 없음",
                     "B의 FS 인플레이 타구는 65개로 GB%·xwOBAcon은 참고 수준"]},
    ],
    "failure_cases": [
        {"pitcher": "input", "pitch_type": "CH",
         "lesson": "A의 CH는 IVB 9.5in로 포심(15.8in)과 수직 차이가 6.3in에 그쳐 xwOBAcon 0.430, RV/100 -1.6. "
                   "새 오프스피드는 포심 대비 IVB 차이 10in 이상을 목표로 해야 함"}
    ],
    "not_recommended": [
        {"pitch_type": "CU", "reason": "B의 CU는 RV/100 +2.5지만 70구(small_sample)·구사율 3.0%로 근거가 불충분"}
    ],
    "data_limitations": ["단일 시즌 관측 데이터 기반이며 인과 효과가 아님",
                         "타자 구성, 구장, 카운트 등 상황 변수는 반영되지 않음"],
}


# 프롬프트 길이 절약: 중복되거나 판단에 덜 중요한 값은 GPT에 보낼 때만 뺀다 (저장되는 payload는 그대로)
_DROP_PITCH_KEYS = {"csw_pct"}
_DROP_FB_KEYS = {"velo_mph", "ivb_in", "hb_in", "spin_rpm"}          # arsenal의 주 패스트볼 항목과 중복
_DROP_TARGET_KEYS = {"source_usage_pct", "source_n_pitches", "source_small_sample"}  # arsenal과 중복


def compact_payload(payload: dict) -> dict:
    out = json.loads(json.dumps(payload))
    out.pop("task", None)
    for who in ("input_pitcher", "similar_pitcher"):
        p = out.get(who, {})
        if p.get("primary_fastball"):
            p["primary_fastball"] = {k: v for k, v in p["primary_fastball"].items() if k not in _DROP_FB_KEYS}
        p["arsenal"] = [{k: v for k, v in a.items() if k not in _DROP_PITCH_KEYS} for a in p.get("arsenal", [])]
    out["transfer_targets"] = [{k: v for k, v in t.items() if k not in _DROP_TARGET_KEYS}
                               for t in out.get("transfer_targets", [])]
    return out


def _dumps(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))   # 공백 없이 (토큰 절약)


def build_messages(payload: dict, one_shot: bool = True) -> list[dict]:
    """system → one-shot(user/assistant) → 실제 요청(user) 순서의 messages.

    one_shot=False: 무료 API의 입력 길이 한도를 넘을 때 예시를 빼고 보낸다 (형식은 시스템 프롬프트의 스키마로 유지).
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if one_shot:
        messages += [
            {"role": "user", "content": _dumps(compact_payload(ONE_SHOT_USER))},
            {"role": "assistant", "content": _dumps(ONE_SHOT_ASSISTANT)},
        ]
    messages.append({"role": "user", "content": _dumps(compact_payload(payload))})
    return messages

In [55]:
print(SYSTEM_PROMPT)

당신은 MLB 투구 설계(pitch design) 분석가입니다.
대학 연구 프로젝트 '주 패스트볼 shape 기반 개인화 변화구 추천'의 마지막 단계에서,
[입력 투수]와 패스트볼이 비슷하지만 성적이 더 좋았던 [유사 투수]의 Statcast 집계 데이터를 비교해
입력 투수에게 맞는 변화구(세컨더리 피치) shape 목표를 최대 3개 제시합니다.

## 입력 데이터 정의
- 모든 수치는 해당 시즌 MLB 정규시즌 Statcast 투구 데이터에서 계산한 값입니다.
- velo_mph: 평균 구속(mph). ivb_in: 수직 무브먼트(인치, 중력 제외, +는 떠오름).
- hb_in: 수평 무브먼트(인치). **암사이드(투수 팔 쪽) = +, 글러브사이드 = −** 로 좌/우완 통일.
- *_gap_vs_fb: 같은 투수의 주 패스트볼 대비 차이(해당 구종 − 주 패스트볼).
- whiff_pct: 헛스윙/스윙(%). chase_pct: 존 밖 공에 대한 스윙 비율(%).
- primary_fastball: 주 패스트볼의 릴리스 기하(팔 각도, 익스텐션, 릴리스 높이/좌우). 구속·무브먼트는 arsenal의 해당 구종 참고.
- rv_per_100: 100구당 투수 기준 런 밸류. **+일수록 투수에게 좋음**. (주 성과지표)
- gb_pct: 인플레이 타구(BBE) 중 땅볼 비율(%). xwobacon: BBE의 기대 wOBA 평균, **낮을수록 투수에게 좋음** (리그 평균 약 0.370 내외).
- n_bbe: 인플레이 타구 수. 30개 미만이면 gb_pct·xwobacon은 불안정합니다.
- small_sample=true: 100구 미만 → 비율 지표가 불안정함.
- transfer_targets: 코드가 미리 계산한 목표 shape = 입력 투수 주 패스트볼 + 유사 투수의 (구종 − 패스트볼) 차이.
- search_context: 유사 투수 검색 단계에서 넘어온 정보(클러스터, 거리, FIP 등). 없을 수도 있습니다.

## 반드시 지킬 규칙


In [56]:
OUTPUT_DIR = ROOT / "data" / "processed" / "output" / "llm_recommendations"
# OpenAI 호환 방식으로 접속할 수 있는 제공자 (OpenAI SDK 그대로 사용, 주소만 다름)
PROVIDERS = {
    "openai": {"key_env": "OPENAI_API_KEY", "base_url": None, "default_model": "gpt-5-mini"},
    "gemini": {"key_env": "GEMINI_API_KEY",
               "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "default_model": "gemini-3.8-flash"},
}
MAX_RETRIES = 4

## 6. 입력 읽기 (2번 단계 결과 / 클러스터 JSON)

In [57]:
_PLAYER_KEYS = ("pitcher", "pitcher_id", "player_id", "mlbam_id", "id", "player_name", "name", "player")
_YEAR_KEYS = ("year", "season", "game_year")


def _pick(d: dict, keys: tuple):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None


def read_pair_json(path: str) -> tuple[dict, dict, dict]:
    """유사 투수 검색 결과 JSON을 읽는다.

    기대 형식 (키 이름은 어느 정도 유연하게 받음):
    {
      "input":   {"pitcher": 543037, "year": 2024},            # 또는 "player_name"
      "similar": {"pitcher": 592450, "year": 2023},
      ... 그 밖의 키(cluster, distance, fip 등)는 search_context로 GPT에 전달
    }
    """
    with open(ROOT / path, encoding="utf-8") as f:          # 레포 루트 기준 상대경로
        raw = json.load(f)

    inp = raw.get("input") or raw.get("input_pitcher") or raw.get("query")
    sim = raw.get("similar") or raw.get("similar_pitcher") or raw.get("match")
    if isinstance(sim, list) and sim and isinstance(sim[0], (dict, list, tuple)):
        sim = sim[0]                   # 여러 명이 오면 가장 가까운 1명 사용
    if inp is None or sim is None:
        raise ValueError("pair JSON에 'input'과 'similar'가 필요합니다.")

    def to_ref(d):
        # ["Shohei Ohtani", 2024] 형식
        if isinstance(d, (list, tuple)) and len(d) == 2:
            return {"player": d[0], "year": int(d[1])}
        # {"Shohei Ohtani": {"2024": {...}}} 형식 (pitcher_clustered.json과 같은 모양)
        if len(d) == 1:
            (k, v), = d.items()
            if isinstance(v, dict) and len(v) == 1 and str(next(iter(v))).isdigit():
                return {"player": k, "year": int(next(iter(v)))}
        player, year = _pick(d, _PLAYER_KEYS), _pick(d, _YEAR_KEYS)
        if player is None or year is None:
            raise ValueError(f"선수/연도 키를 찾을 수 없습니다: {d}")
        return {"player": player, "year": int(year)}

    # 선수/연도 외의 값은 전부 search_context로 (클러스터, 거리, FIP 등)
    context = {k: v for k, v in raw.items() if k not in ("input", "input_pitcher", "query",
                                                        "similar", "similar_pitcher", "match")}
    for label, d in (("input", inp), ("similar", sim)):
        if not isinstance(d, dict) or len(d) == 1:
            continue
        extra = {k: v for k, v in d.items() if k not in _PLAYER_KEYS + _YEAR_KEYS}
        if extra:
            context[label] = extra
    return to_ref(inp), to_ref(sim), context

In [58]:
CLUSTERS_JSON_NAME = "pitcher_clustered.json"


def _is_cluster_json(data) -> bool:
    """{선수: {연도: {average_velocity, cluster}}} 모양인지 확인 (다른 JSON을 잘못 읽지 않도록)."""
    if not isinstance(data, dict) or not data:
        return False
    seasons = next(iter(data.values()))
    if not isinstance(seasons, dict) or not seasons:
        return False
    k, info = next(iter(seasons.items()))
    return str(k).isdigit() and isinstance(info, dict) and "cluster" in info


def load_clusters_json(path: str | None = None) -> dict:
    """클러스터링 결과 JSON을 읽는다. 없어도 추천은 동작한다 (클러스터 정보만 빠짐).

    기법별(K-Means/DBSCAN/계층/GMM)로 파일이 따로 저장되고 최종 선택 전이라 이름이 다를 수 있으므로:
    1) --clusters-json으로 경로를 주면 그 파일 사용
    2) 아니면 data/ 아래에서 pitcher_clustered.json을 찾고
    3) 없으면 이름에 'clust'가 들어간 JSON 중 형식이 맞는 파일이 딱 1개일 때만 사용
       (여러 개면 어느 기법인지 모르므로 사용하지 않고 --clusters-json 지정을 안내)
    """
    if path:
        p = ROOT / path
    else:
        exact = sorted((ROOT / "data").rglob(CLUSTERS_JSON_NAME))
        if exact:
            p = exact[0]
        else:
            valid = []
            for f in sorted((ROOT / "data").rglob("*clust*.json")):
                try:
                    if _is_cluster_json(json.loads(f.read_text(encoding="utf-8"))):
                        valid.append(f)
                except (json.JSONDecodeError, UnicodeDecodeError):
                    pass
            if len(valid) != 1:
                msg = ("찾지 못해" if not valid else
                       f"후보가 {len(valid)}개라 ({', '.join(rel(v) for v in valid)}) 고르지 못해")
                print(f"[안내] 클러스터 JSON을 {msg} 클러스터 정보 없이 진행합니다. "
                      "--clusters-json으로 지정할 수 있습니다.")
                return {}
            p = valid[0]

    data = json.loads(p.read_text(encoding="utf-8"))
    if not _is_cluster_json(data):
        raise ValueError(f"{rel(p)}이 {{선수: {{연도: {{average_velocity, cluster}}}}}} 형식이 아닙니다.")
    print(f"[clusters] {rel(p)} 로드 ({len(data)}명)")
    return data


def lookup_cluster(clusters: dict, player, year: int) -> dict | None:
    """이름 표기 차이(악센트, 'Last, First', Jr.)를 무시하고 {average_velocity, cluster}를 찾는다."""
    if not clusters:
        return None
    targets = _name_variants(str(player))
    for name, seasons in clusters.items():
        if targets & _name_variants(name) and isinstance(seasons, dict):
            info = seasons.get(str(year))
            if info:
                return {"player_name_in_json": name, **info}
    return None

In [59]:
def build_payload(input_ref: dict, similar_ref: dict, search_context: dict | None = None,
                  force_rerun: bool = False, clusters: dict | None = None) -> dict:
    arsenal = load_arsenal(force_rerun=force_rerun)
    search_context = dict(search_context or {})
    clusters = clusters if clusters is not None else load_clusters_json()

    in_cl = lookup_cluster(clusters, input_ref["player"], input_ref["year"])
    sim_cl = lookup_cluster(clusters, similar_ref["player"], similar_ref["year"])
    if in_cl:
        search_context["input_cluster_info"] = in_cl
    if sim_cl:
        search_context["similar_cluster_info"] = sim_cl
    if in_cl and sim_cl and in_cl.get("cluster") != sim_cl.get("cluster"):
        print(f"[주의] 두 투수의 클러스터가 다릅니다 ({in_cl.get('cluster')} vs {sim_cl.get('cluster')}).")

    in_id = resolve_pitcher(arsenal, input_ref["player"], input_ref["year"],
                            velo_hint=(in_cl or {}).get("average_velocity"))
    sim_id = resolve_pitcher(arsenal, similar_ref["player"], similar_ref["year"],
                             velo_hint=(sim_cl or {}).get("average_velocity"))

    in_sum = pitcher_season_summary(arsenal, in_id, input_ref["year"])
    sim_sum = pitcher_season_summary(arsenal, sim_id, similar_ref["year"])
    if in_sum["throws"] != sim_sum["throws"]:
        print(f"[주의] 두 투수의 손잡이가 다릅니다 ({in_sum['throws']} vs {sim_sum['throws']}). "
              "HB는 암사이드 기준으로 통일돼 있어 비교는 가능합니다.")

    return {
        "task": "입력 투수에게 맞는 변화구 shape 목표를 최대 3개 추천",
        "search_context": search_context,
        "input_pitcher": in_sum,
        "similar_pitcher": sim_sum,
        "transfer_targets": transfer_targets(in_sum, sim_sum),
    }

## 7. LLM 호출 (Gemini / OpenAI)
API 키는 레포 루트의 `.env`에서 `python-dotenv`로 읽는다. **코드에 직접 쓰지 않는다.**
OpenAI 공식 SDK를 쓰고, Gemini는 OpenAI 호환 주소로 접속한다 → `.env`만 바꾸면 GPT로 전환.

In [60]:
def resolve_provider() -> tuple[str, str | None]:
    """.env를 읽고 (제공자, API 키)를 정한다. OPENAI_API_KEY가 있으면 OpenAI, 없고 GEMINI_API_KEY가 있으면 Gemini."""
    load_dotenv(ROOT / ".env")                 # 레포 루트의 .env
    if os.getenv("OPENAI_API_KEY"):
        return "openai", os.getenv("OPENAI_API_KEY")
    if os.getenv("GEMINI_API_KEY"):
        return "gemini", os.getenv("GEMINI_API_KEY")
    return "none", None


def default_model() -> str:
    provider, _ = resolve_provider()
    return os.getenv("LLM_MODEL") or PROVIDERS.get(provider, PROVIDERS["gemini"])["default_model"]


def get_client():
    provider, api_key = resolve_provider()
    if not api_key:
        raise RuntimeError(
            "API 키가 없습니다. 레포 루트의 .env에 GEMINI_API_KEY=... (무료) 또는 "
            "OPENAI_API_KEY=... 를 적어주세요. 코드에 직접 쓰지 마세요."
        )
    from openai import OpenAI                   # dry-run 때는 설치 안 돼 있어도 되게 여기서 import
    return OpenAI(api_key=api_key, base_url=PROVIDERS[provider]["base_url"])


def call_gpt(messages: list[dict], client=None, model: str | None = None) -> tuple[dict, str]:
    client = client or get_client()
    model = model or default_model()

    last_err = None
    use_json_mode = True
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            kwargs = {"response_format": {"type": "json_object"}} if use_json_mode else {}   # JSON만 반환하도록 강제
            resp = client.chat.completions.create(model=model, messages=messages, **kwargs)
        except Exception as e:
            msg = str(e).lower()
            if use_json_mode and "response_format" in msg:   # JSON 모드를 지원하지 않는 모델이면 빼고 재시도
                print("[LLM] 이 모델은 JSON 모드를 지원하지 않아 끄고 다시 요청합니다 (형식은 프롬프트로 유지).")
                use_json_mode = False
                continue
            too_long = "token" in msg and any(w in msg for w in ("limit", "too large", "too long", "maximum"))
            if too_long and len(messages) == 4:            # one-shot 포함 상태면 빼고 한 번 더
                print("[LLM] 입력 길이 한도 초과 → one-shot 예시를 빼고 다시 요청합니다.")
                messages = [messages[0], messages[-1]]
                continue
            raise
        text = _response_text(resp)
        try:
            return json.loads(_strip_code_fence(text)), model
        except json.JSONDecodeError as e:
            last_err = e
            print(f"[LLM] JSON 파싱 실패 (시도 {attempt}/{MAX_RETRIES}), 재시도")
    raise RuntimeError(f"GPT 응답을 JSON으로 읽지 못했습니다: {last_err}")


def _response_text(resp) -> str:
    """응답에서 모델이 쓴 텍스트만 꺼낸다.

    보통은 ChatCompletion 객체지만, SDK 버전·서버(GitHub Models 등)에 따라
    JSON 문자열, dict, 스트리밍(data: ...) 문자열로 올 때도 있어 모두 처리한다.
    """
    if isinstance(resp, (bytes, bytearray)):
        resp = resp.decode("utf-8", errors="replace")
    if isinstance(resp, str):
        raw = resp.strip()
        if raw.startswith("data:"):                       # 스트리밍 형식
            parts = []
            for line in raw.splitlines():
                line = line.strip()
                if not line.startswith("data:") or line == "data: [DONE]":
                    continue
                try:
                    chunk = json.loads(line[5:].strip())
                except json.JSONDecodeError:
                    continue
                for ch in chunk.get("choices", []):
                    parts.append((ch.get("delta") or ch.get("message") or {}).get("content") or "")
            return "".join(parts)
        try:
            resp = json.loads(raw)
        except json.JSONDecodeError:
            raise RuntimeError("API가 예상하지 못한 응답을 보냈습니다 (JSON 아님). 앞부분:\n" + raw[:500])
    if isinstance(resp, dict):
        if "error" in resp:
            raise RuntimeError(f"API 오류 응답: {resp['error']}")
        try:
            return resp["choices"][0]["message"]["content"] or ""
        except (KeyError, IndexError, TypeError):
            raise RuntimeError("응답에 choices가 없습니다. 앞부분:\n" + json.dumps(resp, ensure_ascii=False)[:500])
    return resp.choices[0].message.content or ""


def _strip_code_fence(text: str) -> str:
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())

## 8. 응답 검증
데이터에 없는 구종, 계산값과 다른 목표 shape, 수치 없는 근거를 경고로 잡는다.

In [61]:
REQUIRED_KEYS = ("summary", "fastball_comparison", "recommendations", "failure_cases",
                 "not_recommended", "data_limitations")


def validate_response(resp: dict, payload: dict, tol: float = 0.25) -> list[str]:
    """스키마와 근거를 점검하고 경고 목록을 돌려준다 (빈 리스트면 통과)."""
    warnings = [f"필수 키 없음: {k}" for k in REQUIRED_KEYS if k not in resp]

    allowed = {p["pitch_type"] for p in payload["similar_pitcher"]["arsenal"]}
    allowed |= {p["pitch_type"] for p in payload["input_pitcher"]["arsenal"]}
    targets = {t["pitch_type"]: t for t in payload["transfer_targets"]}

    recs = resp.get("recommendations", [])
    if len(recs) > 3:
        warnings.append(f"추천이 {len(recs)}개 (최대 3개)")
    for r in recs:
        pt = r.get("pitch_type")
        if pt not in allowed:
            warnings.append(f"[{pt}] 입력 데이터에 없는 구종을 추천함")
            continue
        if r.get("action") not in ("add", "refine", "usage"):
            warnings.append(f"[{pt}] action 값이 잘못됨: {r.get('action')}")
        if r.get("action") in ("add", "refine") and pt in targets:
            want = targets[pt]["target_shape"]
            got = r.get("target_shape") or {}
            for k, v in want.items():
                if not isinstance(got.get(k), (int, float)) or abs(got[k] - v) > tol:
                    warnings.append(f"[{pt}] target_shape.{k}={got.get(k)} ≠ 계산값 {v}")
        if not r.get("rationale"):
            warnings.append(f"[{pt}] 근거(rationale)가 비어 있음")
        elif not all(re.search(r"\d", s) for s in r["rationale"]):
            warnings.append(f"[{pt}] 수치를 인용하지 않은 근거가 있음")
    return warnings

## 9. 저장 (JSON + Markdown)

In [62]:
def _fmt_shape(s: dict) -> str:
    if not s:
        return "-"
    return f"{s.get('velo_mph')} mph / IVB {s.get('ivb_in')} in / HB {s.get('hb_in')} in"


def to_markdown(resp: dict, payload: dict, model: str, warnings: list[str]) -> str:
    a, b = payload["input_pitcher"], payload["similar_pitcher"]
    lines = [
        f"# 변화구 추천: {a['player_name']} ({a['season']})",
        f"- 비교 대상: **{b['player_name']} ({b['season']})**",
        f"- 모델: `{model}` · 생성: {datetime.now():%Y-%m-%d %H:%M}",
        "- HB는 암사이드 +, RV/100은 투수 기준 + = 좋음",
        "",
        "## 요약", resp.get("summary", ""), "",
        "## 패스트볼 비교",
    ]
    fc = resp.get("fastball_comparison", {})
    lines += [f"- 공통점: {s}" for s in fc.get("similarities", [])]
    lines += [f"- 차이점: {s}" for s in fc.get("differences", [])]
    lines += ["", "## 추천 변화구"]
    for r in resp.get("recommendations", []):
        lines += [
            f"### {r.get('rank')}. {r.get('pitch_name')} ({r.get('pitch_type')}) — {r.get('action')}, 신뢰도 {r.get('confidence')}",
            f"- 목표 shape: {_fmt_shape(r.get('target_shape'))}",
            f"- 패스트볼 대비: {_fmt_shape(r.get('gap_vs_fastball'))}",
            f"- 사용 계획: {r.get('usage_plan', '')}",
            "- 근거:", *[f"  - {s}" for s in r.get("rationale", [])],
            "- 주의:", *[f"  - {s}" for s in r.get("caveats", [])],
            "",
        ]
    if resp.get("failure_cases"):
        lines += ["## 실패 사례 (피해야 할 shape)"]
        lines += [f"- [{x.get('pitcher')}] {x.get('pitch_type')}: {x.get('lesson')}" for x in resp["failure_cases"]]
        lines += [""]
    if resp.get("not_recommended"):
        lines += ["## 추천하지 않은 구종"]
        lines += [f"- {x.get('pitch_type')}: {x.get('reason')}" for x in resp["not_recommended"]]
        lines += [""]
    lines += ["## 데이터 한계"] + [f"- {s}" for s in resp.get("data_limitations", [])]
    if warnings:
        lines += ["", "## ⚠️ 자동 검증 경고"] + [f"- {w}" for w in warnings]
    return "\n".join(lines) + "\n"


def save_outputs(resp: dict, payload: dict, model: str, warnings: list[str]) -> tuple:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    a, b = payload["input_pitcher"], payload["similar_pitcher"]
    stem = f"{a['pitcher_id']}_{a['season']}__vs_{b['pitcher_id']}_{b['season']}"
    json_path, md_path = OUTPUT_DIR / f"{stem}.json", OUTPUT_DIR / f"{stem}.md"
    record = {"model": model, "created_at": datetime.now().isoformat(timespec="seconds"),
              "payload": payload, "response": resp, "validation_warnings": warnings}
    json_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    md_path.write_text(to_markdown(resp, payload, model, warnings), encoding="utf-8")
    return json_path, md_path

## 10. 전체 실행 함수

In [63]:
def recommend(input_ref: dict, similar_ref: dict, search_context: dict | None = None,
              dry_run: bool = False, client=None, model: str | None = None,
              clusters_json: str | None = None) -> dict:
    """2번 단계(유사 투수 검색) 코드에서 바로 호출할 수도 있다:
        recommend({"player": "Shohei Ohtani", "year": 2024}, {"player": "Justin Verlander", "year": 2024})
    """
    payload = build_payload(input_ref, similar_ref, search_context,
                            clusters=load_clusters_json(clusters_json))
    messages = build_messages(payload)

    if dry_run:
        print("=== [dry-run] GPT에 보낼 마지막 user 메시지 ===")
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f"\n(messages {len(messages)}개: system + one-shot user/assistant + 실제 요청)")
        return {"payload": payload, "messages": messages}

    resp, used_model = call_gpt(messages, client=client, model=model)
    warnings = validate_response(resp, payload)
    json_path, md_path = save_outputs(resp, payload, used_model, warnings)
    print(f"저장: {rel(json_path)}\n저장: {rel(md_path)}")
    if warnings:
        print("[검증 경고]\n - " + "\n - ".join(warnings))
    return {"payload": payload, "response": resp, "warnings": warnings,
            "json_path": json_path, "md_path": md_path}

## 11. API 키 확인
`.env`에서 키를 읽었는지만 확인한다 (키 값은 출력하지 않음).

In [64]:
provider, _key = resolve_provider()
if _key:
    print(f"API 키: 설정됨 ({provider})")
    print(" - 접속:", PROVIDERS[provider]["base_url"] or "OpenAI (api.openai.com)")
    print(" - 모델:", MODEL or default_model())
else:
    print("API 키: 없음 → .env에 GEMINI_API_KEY=... 를 넣어야 실제 호출 가능 (DRY_RUN은 가능)")

API 키: 설정됨 (gemini)
 - 접속: https://generativelanguage.googleapis.com/v1beta/openai/
 - 모델: gemini-3.8-flash


## 12. 실행

In [43]:
if PAIR_JSON:
    input_ref, similar_ref, ctx = read_pair_json(PAIR_JSON)
else:
    input_ref = {"player": INPUT_PLAYER, "year": INPUT_YEAR}
    similar_ref = {"player": SIMILAR_PLAYER, "year": SIMILAR_YEAR}
    ctx = {}

if FORCE_RERUN:
    load_arsenal(force_rerun=True)

result = recommend(input_ref, similar_ref, ctx, dry_run=DRY_RUN, model=MODEL,
                   clusters_json=CLUSTERS_JSON)

[안내] 클러스터 JSON을 찾지 못해 클러스터 정보 없이 진행합니다. --clusters-json으로 지정할 수 있습니다.
저장: data/processed/output/llm_recommendations/660271_2023__vs_434378_2024.json
저장: data/processed/output/llm_recommendations/660271_2023__vs_434378_2024.md


## 13. 결과 확인
- 두 투수의 구종 표 (프롬프트에 들어간 수치)
- `DRY_RUN = False`면 GPT 추천 결과(Markdown)와 검증 경고

In [44]:
payload = result["payload"]
cols = ["pitch_type", "usage_pct", "velo_mph", "ivb_in", "hb_in", "velo_gap_vs_fb", "ivb_gap_vs_fb",
        "hb_gap_vs_fb", "whiff_pct", "chase_pct", "gb_pct", "xwobacon", "rv_per_100", "n_pitches"]
for key in ("input_pitcher", "similar_pitcher"):
    p = payload[key]
    print(f"[{key}] {p['player_name']} ({p['season']}, {p['throws']}HP)")
    display(pd.DataFrame(p["arsenal"])[cols])

print("[transfer_targets] 코드가 계산한 목표 shape")
display(pd.json_normalize(payload["transfer_targets"]))

if "md_path" in result:
    display(Markdown(result["md_path"].read_text(encoding="utf-8")))

[input_pitcher] Ohtani, Shohei (2023, RHP)


,pitch_type,usage_pct,velo_mph,ivb_in,hb_in,velo_gap_vs_fb,ivb_gap_vs_fb,hb_gap_vs_fb,whiff_pct,chase_pct,gb_pct,xwobacon,rv_per_100,n_pitches
0,ST,35.0,83.7,3.5,-15.9,-13.1,-10.6,-20.0,35.9,31.8,36.7,0.34,1.22,733
1,FF,32.9,96.8,14.1,4.1,0.0,0.0,0.0,20.0,19.6,42.9,0.38,1.48,688
2,FC,12.2,89.5,6.2,-3.7,-7.3,-7.9,-7.8,18.2,25.7,47.8,0.43,-0.30,255
3,SI,6.5,94.4,5.7,14.7,-2.4,-8.4,10.6,18.2,13.9,71.4,0.31,-1.89,137
4,FS,5.8,88.2,3.7,7.6,-8.6,-10.5,3.5,44.2,31.6,73.7,0.29,0.10,121
5,SL,3.9,85.3,-1.4,-5.6,-11.5,-15.5,-9.7,30.8,33.3,46.2,0.75,-4.02,82
6,CU,3.7,75.7,-13.4,-11.9,-21.1,-27.5,-16.0,41.7,17.4,44.4,0.33,1.62,78


[similar_pitcher] Verlander, Justin (2024, RHP)


,pitch_type,usage_pct,velo_mph,ivb_in,hb_in,velo_gap_vs_fb,ivb_gap_vs_fb,hb_gap_vs_fb,whiff_pct,chase_pct,gb_pct,xwobacon,rv_per_100,n_pitches
0,FF,48.5,93.5,19.3,8.6,0.0,0.0,0.0,16.5,28.8,27.9,0.35,-0.65,767
1,CU,21.9,77.6,-13.4,-7.2,-15.8,-32.7,-15.9,21.2,17.4,33.3,0.39,-1.16,346
2,SL,19.3,86.7,5.9,-4.0,-6.8,-13.4,-12.6,20.2,28.5,29.6,0.28,0.79,305
3,CH,10.1,84.0,9.7,13.4,-9.5,-9.6,4.8,25.3,35.7,24.0,0.33,-0.32,160


[transfer_targets] 코드가 계산한 목표 shape


,pitch_type,pitch_name,source_usage_pct,source_n_pitches,source_small_sample,input_already_throws,gap_vs_fb.velo_mph,gap_vs_fb.ivb_in,gap_vs_fb.hb_in,target_shape.velo_mph,target_shape.ivb_in,target_shape.hb_in,input_current_shape.velo_mph,input_current_shape.ivb_in,input_current_shape.hb_in
0,CU,Curveball,21.9,346,False,True,-15.8,-32.7,-15.9,81.0,-18.6,-11.8,75.7,-13.4,-11.9
1,SL,Slider,19.3,305,False,True,-6.8,-13.4,-12.6,90.0,0.7,-8.5,85.3,-1.4,-5.6
2,CH,Changeup,10.1,160,False,False,-9.5,-9.6,4.8,87.3,4.5,8.9,NaN,NaN,NaN


# 변화구 추천: Ohtani, Shohei (2023)
- 비교 대상: **Verlander, Justin (2024)**
- 모델: `gemini-3.8-flash` · 생성: 2026-09-26 12:22
- HB는 암사이드 +, RV/100은 투수 기준 + = 좋음

## 요약
오타니와 벌랜더는 우완 주 패스트볼로 포심을 구사하지만, 릴리스 높이(5.7ft vs 7.1ft)와 팔 각도(36.4° vs 56.6°)에서 뚜렷한 차이가 있습니다. 벌랜더의 구종 중 유일하게 양수 런 밸류를 기록한 하드 슬라이더(SL)의 shape를 차용해 오타니의 부진했던 기존 SL을 재설계(refine)하는 방안을 제시합니다.

## 패스트볼 비교
- 공통점: 모두 93mph 이상의 강한 포심 구속(오타니 96.8mph vs 벌랜더 93.5mph)과 2200~2400rpm대의 회전수(2259 vs 2395 rpm)를 기록
- 공통점: 좌/우타자 상대 포심 구사율을 비교적 균등하게 배분(오타니 좌 32.5%/우 33.2%, 벌랜더 좌 47.0%/우 50.2%)
- 차이점: 팔 각도(36.4° vs 56.6°)와 릴리스 높이(5.7ft vs 7.1ft)의 차이로 수직 투구 궤적이 크게 다름
- 차이점: 포심 IVB(14.1in vs 19.3in)와 HB(4.1in vs 8.6in)에서 벌랜더가 더 높은 수직/수평 무브먼트를 형성
- 차이점: 포심 성과 지표에서 오타니는 RV/100 +1.48로 위력적이었으나 벌랜더는 -0.65에 그침

## 추천 변화구
### 1. Slider (SL) — refine, 신뢰도 medium
- 목표 shape: 90.0 mph / IVB 0.7 in / HB -8.5 in
- 패스트볼 대비: -6.8 mph / IVB -13.4 in / HB -12.6 in
- 사용 계획: 벌랜더처럼 우타자(22.9%)와 좌타자(16.0%)를 가리지 않고 카운트 싸움 및 약한 컨택 유도용으로 활용
- 근거:
  - 벌랜더의 SL은 RV/100 +0.79, xwOBAcon 0.284(71 BBE)로 벌랜더 아스날 중 유일하게 정타 억제와 양수 런 밸류를 기록
  - 오타니의 기존 SL(85.3mph, IVB -1.4in, HB -5.6in)은 xwOBAcon 0.754, RV/100 -4.02로 극도로 취약했음
  - 목표 shape(90.0mph, IVB 0.7in, HB -8.5in)는 구속을 끌어올리고 포심 대비 구속 차이를 -6.8mph로 좁혀 89.5mph FC와 83.7mph ST 사이의 단단한 브릿지 구종 역할을 수행 가능
- 주의:
  - 두 투수의 팔 각도(36.4° vs 56.6°) 격차가 커서 벌랜더와 동일한 상대적 무브먼트 갭 구현에 생체역학적 제약이 있을 수 있음
  - 오타니가 이미 커터(FC, 12.2%)와 스위퍼(ST, 35.0%)를 높은 비중으로 던지고 있어 구종 간 역할 중복 가능성 존재

## 실패 사례 (피해야 할 shape)
- [input] SL: 오타니의 기존 SL은 85.3mph의 애매한 구속과 어중간한 무브먼트(IVB -1.4in, HB -5.6in)로 인해 xwOBAcon 0.754, RV/100 -4.02를 기록하며 장타를 허용함
- [similar] CU: 벌랜더의 CU는 21.9%의 높은 구사율에도 불구하고 RV/100 -1.16, xwOBAcon 0.391을 기록하여 큰 수직 낙차(IVB -13.4in) 대비 타구 억제 성과가 부진했음

## 추천하지 않은 구종
- CU: 벌랜더의 CU는 RV/100 -1.16으로 성과가 나빴으며, 오타니는 이미 Whiff% 41.7%, RV/100 +1.62의 효율적인 CU를 보유하고 있음
- CH: 벌랜더의 CH는 RV/100 -0.32로 음수를 기록했고, 오타니는 이미 좌타자 대항마로 위력적인 FS(Whiff% 44.2%, GB% 73.7%, xwOBAcon 0.286)를 갖추고 있어 불필요함

## 데이터 한계
- 벌랜더의 릴리스 높이(7.1ft)와 오타니(5.7ft)의 차이가 1.4ft에 달해 단순 shape 이전의 물리적 한계가 존재함
- 오타니의 SL(82구), CU(78구) 등 일부 구종은 100구 미만의 small_sample 데이터로 지표 변동성이 큼
- 볼카운트별 사용 빈도나 타자의 반응 등 세부 경기 맥락은 포함되지 않음
